# **＜重要＞本章で使用するファイル**

- ab_purchase.csv
- ab_test.csv
- ab_web_access.csv
- ec_data.csv
- product_master.csv
- purchase.csv
- web_access.csv

これのファイルを左サイドバーのファイルからアップロードしてお使いください。（p.20参照）

Google ドライブにファイルを置き、マウントして使うこともできます（p.21参照）。その場合は次の点に注意してください。

- ファイルのパスは、各自のフォルダに合わせて書き換えてください
- ドライブ上のファイルは各自で管理してください
- ファイルを書き込むコード（to_csv() など）を実行すると、パスを誤ったときに既存のファイルを上書きするおそれがあります

# Menu.40 データの基本情報を確認しよう

基本情報を確認する

In [ ]:
import pandas as pd

df = pd.read_csv("ec_data.csv")

print(df.head())
print(df.info())

基本統計量を確認する

In [ ]:
print(df.describe())

## Column 実務で発生する前処理

商品に関連するログを抽出

In [ ]:
import pandas as pd

df_access = pd.read_csv("web_access.csv")
df_product = pd.read_csv("product_master.csv")

print(len(df_access)) # 絞り込み前の行数

# 商品に関連するページ(product_idが空でない行)だけを残す
df_access = df_access[df_access["product_id"].notna()]
print(len(df_access)) # 絞り込み後の行数
print(df_access.head())

商品ページの閲覧セッション数を数える

In [ ]:
# 商品ページを見たセッション数
detail_view = df_access.groupby("product_id")["session_id"].nunique().rename("detail_view_sessions")
print(detail_view.head())

レビュー閲覧・購入のセッション数を数える

In [ ]:
# レビュー閲覧セッション数(review_listページに絞ってから数える)
review_logs = df_access[df_access["page_type"] == "review_list"]
review_view = review_logs.groupby("product_id")["session_id"].nunique().rename("review_view_sessions")

# 購入セッション数(purchase_completeページに絞ってから数える)
purchase_logs = df_access[df_access["page_type"] == "purchase_complete"]
purchase = purchase_logs.groupby("product_id")["session_id"].nunique().rename("purchase_sessions")

3つの集計を1つにまとめる

In [ ]:
ec_data = pd.concat([detail_view, review_view, purchase], axis=1)
print(ec_data.head())

欠損値を0で埋めて整数にする

In [ ]:
ec_data = ec_data.fillna(0).astype(int)
print(ec_data.head())

閲覧率・購入率を算出して商品マスタと結合する

In [ ]:
ec_data["review_view_rate"] = ec_data["review_view_sessions"] / ec_data["detail_view_sessions"]
ec_data["conversion_rate"] = ec_data["purchase_sessions"] / ec_data["detail_view_sessions"]

ec_data = ec_data.reset_index().merge(
    df_product[["product_id", "category", "base_price"]],
    on="product_id"
)

print(ec_data.head())

# Menu.41 相関分析で関係性を探ろう

相関行列を算出する

In [ ]:
import pandas as pd

ec_data = pd.read_csv("ec_data.csv")

# 数値列の相関行列を計算
correlation_matrix = ec_data.corr(numeric_only=True)
print(correlation_matrix)

# Menu.42 相関をグラフで読み解こう

相関行列のヒートマップを作成する

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 相関行列の計算
ec_data = pd.read_csv("ec_data.csv")
corr_matrix = ec_data.corr(numeric_only=True)

# ヒートマップの描画
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix,
            annot=True,
            cmap="coolwarm",
            fmt=".2f",
            center=0)
plt.show()

散布図マトリクスを作成する

In [ ]:
# 散布図マトリクスの描画
cols = [ "base_price", "detail_view_sessions", "review_view_sessions", "purchase_sessions", "review_view_rate", "conversion_rate"]
sns.pairplot(ec_data[cols])
plt.show()

# Menu.43 相関をもとに施策を立て、A/Bテストを実施しよう

基本情報を確認する

In [ ]:
import pandas as pd

df_ab_test = pd.read_csv("ab_test.csv")

print(df_ab_test.head())
print(df_ab_test.info())

グループごとの平均値と購入者のみの平均値を算出する

In [ ]:
# グループごとの平均値を算出
summary = df_ab_test.groupby("group")[["review_view_flag", "purchase_flag", "purchase_amount"]].mean()
print("グループ別の集計結果")
print(summary)

# ARPPU(購入者のみの平均購入金額)を計算
arppu = df_ab_test[df_ab_test["purchase_amount"] > 0].groupby("group")["purchase_amount"].mean()
print("ARPPU(購入者のみの平均購入金額)")
print(arppu)

## Column データ準備：生データの確認と集計

データを読み込んで中身を確認する

In [ ]:
import pandas as pd

df_web = pd.read_csv("ab_web_access.csv")
df_purchase = pd.read_csv("ab_purchase.csv")

print(len(df_web), len(df_purchase))
print(df_web.head())

セッション単位にまとめる

In [ ]:
session_agg = df_web.groupby("session_id").agg(
    user_id=("user_id", "first"),
    group=("group", "first")
).reset_index()
print(len(session_agg))
print(session_agg.head())

レビューを見たかどうかのフラグを立てる

In [ ]:
review_sessions = df_web[df_web["page_type"] == "review_list"]["session_id"].unique()
session_agg["review_view_flag"] = session_agg["session_id"].isin(review_sessions)
print(session_agg.head())

購入金額を結合する

In [ ]:
purchase_agg = df_purchase.groupby("session_id")["purchase_amount"].sum().reset_index()
df_merged = session_agg.merge(purchase_agg, on="session_id", how="left")
print(df_merged.head())

金額を整えてフラグを作り、保存する

In [ ]:
df_merged["purchase_amount"] = df_merged["purchase_amount"].fillna(0).astype(int)
df_merged["purchase_flag"] = df_merged["purchase_amount"] > 0
ab_test = df_merged[["session_id", "user_id", "group", "review_view_flag", "purchase_flag", "purchase_amount"]]
ab_test.to_csv("ab_test.csv", index=False)
print(ab_test.head())

# Menu.45 カイ二乗検定で割合の差を評価しよう

クロス集計表を作成する

In [ ]:
import pandas as pd

df_ab = pd.read_csv("ab_test.csv")

# クロス集計表(購入した人数、しなかった人数)の作成
ct = df_ab.pivot_table(index="group", columns="purchase_flag", aggfunc="size")
print(ct)

カイ二乗統計量・p値・期待度数を算出する

In [ ]:
from scipy import stats

# カイ二乗検定
chi2, p, dof, expected = stats.chi2_contingency(ct)

print("カイ二乗統計量:", chi2)
print("p値:", p)
print("期待度数:")
print(expected)

# Menu.46 t検定で平均の差を評価しよう

t検定を実行する

In [ ]:
import pandas as pd
from scipy import stats

df_ab = pd.read_csv("ab_test.csv")

# 購入者のみのデータを抽出
a_arppu = df_ab[(df_ab["group"] == "A") & (df_ab["purchase_amount"] > 0)]["purchase_amount"]
b_arppu = df_ab[(df_ab["group"] == "B") & (df_ab["purchase_amount"] > 0)]["purchase_amount"]

# t検定(Welchのt検定: equal_var=False)
t, p = stats.ttest_ind(b_arppu, a_arppu, equal_var=False)

print("AのARPPU:", a_arppu.mean())
print("BのARPPU:", b_arppu.mean())
print("p値 =", p)

# Menu.47 信頼区間で効果の確からしさの範囲を測ろう

グループごとに集計する

In [ ]:
import pandas as pd
df_ab = pd.read_csv("ab_test.csv")

# グループごとの件数、購入数、購入率を集計
summary = df_ab.groupby("group")["purchase_flag"].agg(["count", "sum", "mean"])
summary.columns = ["n", "purchases", "purchase_rate"]
print(summary)

購入率の差を算出する

In [ ]:
# それぞれのサンプルサイズ、購入数、購入率
n_a, x_a, p_a = summary.loc["A", ["n", "purchases", "purchase_rate"]]
n_b, x_b, p_b = summary.loc["B", ["n", "purchases", "purchase_rate"]]

# 購入率の差(点推定値)
diff = p_b - p_a
print("購入率の差 (B-A):", diff)

95%信頼区間を計算する

In [ ]:
from statsmodels.stats.proportion import confint_proportions_2indep

# 95%信頼区間(差 = B - A)
ci_low, ci_high = confint_proportions_2indep(
    count1=x_b, nobs1=n_b, # グループBの購入数とサンプルサイズ
    count2=x_a, nobs2=n_a, # グループAの購入数とサンプルサイズ
    method="newcomb" # Newcombe法(低い割合でも安定)
)

print("購入率の差:", round(diff, 4))
print("95%信頼区間:", [round(float(ci_low), 4), round(float(ci_high), 4)])